# Catch the Eavesdropper (BB84)

*Secure the Grid, step 1 · UTC Problem 3*

Share a secret key over a simulated fiber link, then add an eavesdropper and catch them by the errors they cause.

**Your job:** fill in the encoding step and the eavesdropper, then report the error rate (QBER) with and without them.

> **Before you run anything, pick the kernel.** When this notebook opens, a **Select Kernel** box appears with *Python 3 (ipykernel)* already selected. That's the wrong one: change it to **qBraid-SDK (v0.12.2)**, then click **Select**. Picked the wrong one, or closed the box? Click the kernel name at the top right of the notebook and choose **qBraid-SDK (v0.12.2)**. The first cell checks, and tells you if it's wrong.

In [7]:
%pip install qbraid matplotlib numpy qiskit qiskit-aer==0.18.2

Note: you may need to restart the kernel to use updated packages.


In [8]:
# Use the qBraid-SDK (v0.12.2) kernel. This stops with a clear message if another one is picked.
import importlib.util
missing = [m for m in ("qbraid", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("Wrong kernel: this one has no " + " or ".join(missing) + ". Click the kernel name "
                      "at the top right of the notebook, pick qBraid-SDK (v0.12.2), then run this cell again.")

# Qiskit Aer is Qiskit's simulator. It runs right here in your Lab instance: instant, free and noise-free,
# with no job and no credits. The first run installs it (a few seconds), pinned to the version tested here.
try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer==0.18.2

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
sim = AerSimulator()
rng = np.random.default_rng(42)

n_bits = 200          # number of qubits Alice sends (a few seconds to run)

print("Setup OK: the qBraid kernel and Qiskit Aer are ready. You're good to go.")

Setup OK: the qBraid kernel and Qiskit Aer are ready. You're good to go.


In [9]:
# Alice's random bits and random bases (0 = Z/standard, 1 = X/Hadamard)
alice_bits  = rng.integers(0, 2, n_bits)
alice_bases = rng.integers(0, 2, n_bits)
bob_bases   = rng.integers(0, 2, n_bits)


In [10]:

def encode(bit, basis):
    qc = QuantumCircuit(1, 1)

    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)

    return qc



In [11]:
def measure(qc, basis):
    """Measure a 1-qubit circuit in the given basis; return the classical bit."""
    if basis == 1:
        qc.h(0)                     # rotate X basis back to Z before measuring
    qc.measure(0, 0)
    counts = sim.run(qc, shots=1).result().get_counts()   # Aer runs x, h and measure as is: no transpile, much faster
    return int(max(counts, key=counts.get))


In [12]:

def run_bb84(eavesdrop=False):
    bob_results = []
    for i in range(n_bits):
        qc = encode(alice_bits[i], alice_bases[i])

        if eavesdrop:
            eve_basis = rng.integers(0, 2)
            eve_bit = measure(qc, eve_basis)
            qc = encode(eve_bit, eve_basis)

        bob_results.append(measure(qc, bob_bases[i]))

    return np.array(bob_results)


In [13]:
def qber(bob_results):
    """Sift (keep matching bases) and return the error rate on the shared key.
    Real BB84 compares only a random sample of the sifted bits in public, then throws them away;
    here we compare every bit so the numbers are easy to see."""
    keep = alice_bases == bob_bases
    a = alice_bits[keep]
    b = bob_results[keep]
    if len(a) == 0:
        return 0.0
    return np.mean(a != b)

clean = run_bb84(eavesdrop=False)
print("QBER, no eavesdropper:", round(qber(clean), 3))

eve = run_bb84(eavesdrop=True)
print("QBER, with eavesdropper:", round(qber(eve), 3), "  (expect about 0.25; with 200 rounds, 0.18 to 0.33 is normal)")


QBER, no eavesdropper: 0.0
QBER, with eavesdropper: 0.215   (expect about 0.25; with 200 rounds, 0.18 to 0.33 is normal)


### Hardware run: one job, not hundreds

The simulator cells above send one qubit per run, which is fine on a free local simulator. On real hardware that would be hundreds of queued, billed jobs. Instead, put 16 BB84 rounds side by side in one circuit (qubit *k* carries round *k*) and run it once with 200 shots.

Every shot repeats the same 16 rounds. That's fine here: Part 3 asks for the error rate caused by hardware noise, and 200 shots × the sifted rounds (6 of the 16 with this seed) give you about 1,200 measured bits to estimate it. They're 200 repeats of the same few states on the same few qubits, so treat the result as a rough estimate.

In [14]:
def bb84_batch_circuit(bits, bases, bob_bases):
    """One circuit with one BB84 round per qubit. Reuses your encode() from above."""
    k = len(bits)
    qc = QuantumCircuit(k, k)
    for q in range(k):
        qc.compose(encode(bits[q], bases[q]), qubits=[q], clbits=[q], inplace=True)
        if bob_bases[q] == 1:
            qc.h(q)                 # Bob measures in the X basis
    qc.measure(range(k), range(k))
    return qc


def qber_from_counts(counts, bits, bases, bob_bases):
    """Error rate on the sifted rounds, over every shot."""
    keep = [q for q in range(len(bits)) if bases[q] == bob_bases[q]]
    errors = total = 0
    for bitstring, c in counts.items():
        bob = bitstring.zfill(len(bits))[::-1]   # Qiskit order: qubit 0 is the LAST character
        for q in keep:
            errors += c * (int(bob[q]) != bits[q])
            total += c
    return errors / total


k = 16                              # rounds per circuit = qubits used
hw_args = (alice_bits[:k], alice_bases[:k], bob_bases[:k])
hw_qc = bb84_batch_circuit(*hw_args)

# Check it on the local simulator first (free)
counts = sim.run(transpile(hw_qc, sim), shots=200).result().get_counts()
print("Batched QBER on the local simulator:", round(qber_from_counts(counts, *hw_args), 3))

Batched QBER on the local simulator: 0.0


### Run it on a quantum computer through qBraid

Practice first on IonQ's simulator. It's free, and its `noise_model` option adds IonQ's gate noise. That noise sits mostly on two-qubit gates, so a circuit with few of them (like this one) comes back almost noise-free; real hardware adds readout error on top. Quantinuum's free H2 emulator models readout error too: the cell has the lines to try it. When your circuit works, switch the two lines to the real IonQ QPU for your hardware run. That run uses credits, so cap it at 200 shots and copy the job ID into your write-up.

In [15]:
# --- Run on a quantum computer through qBraid ---
from qbraid.runtime import QbraidProvider
import warnings
warnings.filterwarnings("ignore", message="Circuit contains measurement gates")  # IonQ measures every qubit anyway

provider = QbraidProvider()   # in qBraid Lab this uses your account automatically

# 1) Practice for free: IonQ's simulator with an approximate noise model. This uses Aria's: with 16 qubits,
#    "forte-1" can wait about 10 minutes in IonQ's queue, and BB84 has no two-qubit gates, so both give ~0 errors
device = provider.get_device("ionq:ionq:sim:simulator")
job = device.run(hw_qc, shots=200, runtime_options={"noise_model": "aria-1"})

# 2) Your real hardware run (uses credits). Swap these in when you're ready:
# device = provider.get_device("ionq:ionq:qpu:forte-1")    # or "aws:ionq:qpu:forte-1" if this one shows as unavailable
# job = device.run(hw_qc, shots=200)

# 3) Also free: Quantinuum's H2 emulator. It models a real trapped-ion computer's noise, readout
#    error included. Accept Quantinuum's terms once first (Devices > Quantinuum in your qBraid account).
#    It's an emulator: label its results that way, not as hardware.
# device = provider.get_device("quantinuum:quantinuum:sim:h2-emulator")
# job = device.run(hw_qc, shots=200)

# 4) Also free: qBraid's own QIR simulator. It's noise-free, so it shows the error rate of a perfect channel.
# device = provider.get_device("qbraid:qbraid:sim:qir-sv")
# job = device.run(hw_qc, shots=200)

print("Job ID:", job.id)          # copy this into your write-up
hw_counts = job.result().data.get_counts()
print("Most common results:", dict(sorted(hw_counts.items(), key=lambda kv: -kv[1])[:5]))
print("QBER from this run:", round(qber_from_counts(hw_counts, *hw_args), 3))

Job ID: ionq:ionq:sim:simulator-0b8e-qjob-6af7c050f15852ed27b71f51
Most common results: {'1010010001000110': 3, '1010010010100111': 3, '0010110000100110': 2, '0010111011000111': 2, '0011010001100110': 2}
QBER from this run: 0.0


## What to have ready
Write your answers in `WRITEUP.md` in this folder (see "Due Sunday" in the track README).
- **Part 1:** near-zero QBER with no eavesdropper (your sifted keys match).
- **Part 2:** the QBER with and without the eavesdropper.
- **Part 3:** your hardware QBER (and the Quantinuum emulator's, if you ran it), your threshold reasoning, and the job IDs.
- **Part 4:** the written physics explanation.
- **For your deck:** the error-rate jump is your evidence that physics can catch a spy. What does it mean for the utility?


In [16]:

print("===== BB84 RESULTS =====")
print("Qubits transmitted:", n_bits)

keep = alice_bases == bob_bases
print("Sifted key length:", int(np.sum(keep)))

print("QBER without Eve:", f"{qber(clean):.2%}")
print("QBER with Eve:", f"{qber(eve):.2%}")

print("QBER increase:",
      f"{qber(eve) - qber(clean):.2%}")

print("Batched local QBER:",
      f"{qber_from_counts(counts, *hw_args):.2%}")

print("Alice and Bob match without Eve:",
      np.array_equal(alice_bits[keep], clean[keep]))


===== BB84 RESULTS =====
Qubits transmitted: 200
Sifted key length: 93
QBER without Eve: 0.00%
QBER with Eve: 21.51%
QBER increase: 21.51%
Batched local QBER: 0.00%
Alice and Bob match without Eve: True
